# InfoCSE on STSB

Official paper: https://aclanthology.org/2022.findings-emnlp.223/  
Official code: https://github.com/caskcsg/sentemb/tree/main/InfoCSE  
Checkpoint: `ffgcc/InfoCSE-bert-base`  
Dataset source: mteb/stsbenchmark-sts (https://aclanthology.org/S17-2001/)

This notebook **re-evaluates the authors' released pretrained checkpoint** on one public test set. It does not retrain the original method. It uses a common direct-cosine protocol: encode each sentence independently, compute pairwise cosine similarity, then compare with human scores using Spearman and Pearson correlation. It requires an NVIDIA CUDA GPU and keeps all code in this notebook. Its outputs are empty until you run all cells on the RTX 3080.


In [ ]:
# Install only missing packages; preserve an existing CUDA-enabled PyTorch build.
import importlib.util
import subprocess
import sys

requirements = {
    "torch": "torch>=2.2,<3",
    "transformers": "transformers==4.46.3",
    "datasets": "datasets==3.2.0",
    "huggingface_hub": "huggingface_hub>=0.26,<1",
    "numpy": "numpy>=1.26,<3",
    "scipy": "scipy>=1.11,<2",
    "pandas": "pandas>=2.1,<3",
}
missing = [spec for module, spec in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import importlib.metadata as metadata
for module in requirements:
    package = "huggingface-hub" if module == "huggingface_hub" else module
    print(f"{package}=={metadata.version(package)}")


In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import HfApi
from scipy.stats import pearsonr, spearmanr
from transformers import AutoModel, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

if not torch.cuda.is_available():
    raise RuntimeError("This notebook is configured for a CUDA GPU. Check the RTX 3080 driver and CUDA-enabled PyTorch installation.")
DEVICE = torch.device("cuda:0")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA runtime:", torch.version.cuda)
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

# Batch size 32 and dynamic padding fit typical 10/12 GiB RTX 3080 cards.
BATCH_SIZE = 32
MAX_LENGTH = 512


In [ ]:
METHOD_NAME = 'InfoCSE'
DATASET_NAME = 'STSB'
MODEL_ID = 'ffgcc/InfoCSE-bert-base'
POOLING = 'cls_before_pooler'
EXPECTED_TEST_ROWS = 1379
LABEL_RANGE = (0.0, 5.0)


In [ ]:
if DATASET_NAME == "STSB":
    raw = load_dataset("mteb/stsbenchmark-sts", split="test")
    frame = pd.DataFrame({
        "sentence1": raw["sentence1"],
        "sentence2": raw["sentence2"],
        "gold_score": raw["score"],
    })
else:
    # JSICK mirrors the original English SICK sentences/scores and preserves the standard test split.
    raw = load_dataset("hpprc/jsick", "base", split="test")
    frame = pd.DataFrame({
        "sentence1": raw["premise_en"],
        "sentence2": raw["hypothesis_en"],
        "gold_score": raw["score_en"],
    })

assert len(frame) == EXPECTED_TEST_ROWS, f"Unexpected test set size: {len(frame)}"
assert frame.notna().all().all(), "Dataset contains missing values"
assert frame["gold_score"].between(*LABEL_RANGE).all(), "Score outside documented range"
assert frame["sentence1"].str.len().gt(0).all()
assert frame["sentence2"].str.len().gt(0).all()
print("Dataset:", DATASET_NAME, "| test pairs:", len(frame))
print("Gold score range:", (frame.gold_score.min(), frame.gold_score.max()))
display(frame.head(5))


In [ ]:
api = HfApi()
model_sha = api.model_info(MODEL_ID).sha
dataset_repo = "mteb/stsbenchmark-sts" if DATASET_NAME == "STSB" else "hpprc/jsick"
dataset_sha = api.dataset_info(dataset_repo).sha
print("Model:", MODEL_ID, "| revision:", model_sha)
print("Dataset:", dataset_repo, "| revision:", dataset_sha)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=model_sha)
model = AutoModel.from_pretrained(MODEL_ID, revision=model_sha).to(DEVICE)
model.eval()
assert model.config.hidden_size == 768, "Expected a BERT-base-sized checkpoint"


In [ ]:
def render_text(text):
    if POOLING != "prompt_mask":
        return text
    # Matches the author's unsup-bert evaluation template after placeholder expansion.
    # The original evaluator adds a period when the source has no terminal punctuation.
    if text and text[-1] not in '.?"\'':
        text += "."
    return f'This sentence of "{text}" means{tokenizer.mask_token}.'


@torch.inference_mode()
def encode_sentences(texts):
    vectors = []
    for start in range(0, len(texts), BATCH_SIZE):
        batch_texts = [render_text(text) for text in texts[start:start + BATCH_SIZE]]
        batch = tokenizer(batch_texts, padding=True, truncation=True,
                          max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
        outputs = model(**batch, return_dict=True)
        if POOLING == "cls_before_pooler":
            embeddings = outputs.last_hidden_state[:, 0, :]
        elif POOLING == "cls_pooler":
            embeddings = outputs.pooler_output
            if embeddings is None:
                raise RuntimeError("This checkpoint has no CLS pooler output")
        elif POOLING == "prompt_mask":
            mask_positions = batch["input_ids"].eq(tokenizer.mask_token_id)
            if not torch.all(mask_positions.sum(dim=1) == 1):
                raise RuntimeError("Prompt template must contain exactly one untruncated [MASK]")
            embeddings = outputs.last_hidden_state[mask_positions]
        else:
            raise ValueError(POOLING)
        vectors.append(F.normalize(embeddings.float(), p=2, dim=1).cpu())
    return torch.cat(vectors).numpy()


# Encode each distinct sentence once. This does not use any gold labels.
unique_sentences = list(dict.fromkeys(frame.sentence1.tolist() + frame.sentence2.tolist()))
print("Distinct sentences to encode:", len(unique_sentences))
vectors = encode_sentences(unique_sentences)
vector_by_sentence = dict(zip(unique_sentences, vectors))
left = np.stack([vector_by_sentence[text] for text in frame.sentence1])
right = np.stack([vector_by_sentence[text] for text in frame.sentence2])
frame["cosine_similarity"] = np.sum(left * right, axis=1)
assert np.isfinite(frame["cosine_similarity"]).all()


In [ ]:
gold = frame["gold_score"].to_numpy(dtype=float)
pred = frame["cosine_similarity"].to_numpy(dtype=float)
spearman = spearmanr(gold, pred).statistic
pearson = pearsonr(gold, pred).statistic
summary = pd.DataFrame([{
    "method": METHOD_NAME,
    "dataset": DATASET_NAME,
    "test_pairs": len(frame),
    "spearman": float(spearman),
    "pearson": float(pearson),
    "model_id": MODEL_ID,
    "model_revision": model_sha,
    "dataset_revision": dataset_sha,
    "pooling": POOLING,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
}])
display(summary)
print("Examples with low model similarity:")
display(frame.sort_values("cosine_similarity").head(5))
print("Examples with high model similarity:")
display(frame.sort_values("cosine_similarity", ascending=False).head(5))
